# Random Forest & XGBoost on Tabular Data

**Week 5 · Lesson 8** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week5/lectures/08_random_forest_xgboost_churn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 8.3 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which customers are likely to churn — and how much
predictive lift is on the table if we're willing to trade Lesson 3's
plain-English tree for an ensemble, and then tune it properly?

**Why it matters:** this is the fourth time this dataset has answered that
question, each time with a different model: Week 1's black-box baseline, Week 3
Lesson 5's logistic regression, Week 5 Lesson 3's single pruned tree. Lessons 4-6
built the theory for two ways to combine many trees — **bagging** (Lesson 5,
Random Forest) to fix a tree's variance, **boosting** (Lesson 6, XGBoost) to fix
its bias — and Lesson 7 mapped the knobs that control boosting. This notebook fits
both for real, first at their textbook defaults and then tuned, on the same
churn numbers, lined up against every model that came before.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as every prior lesson that has touched churn — **Telco Customer
Churn**, one row per customer, account/contract/service features plus a binary
`Churn` label. `load_churn()` in `gbus564_coursework.utils.course_data` loads and cleans it the
same way every time.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(exclude="number"))

### Data Prep

Same split as Lesson 3 — **identical** `test_size`, `random_state`, and
`stratify`, on the same `load_churn()` output — so every accuracy number in this
notebook lines up directly against Lesson 3's reported 73.46% baseline and 78.64%
pruned tree. Trees don't need scaling (Lesson 3); the categoricals still need
one-hot encoding. `make_preprocessor()` returns a **fresh** encoder for each
model below, rather than one shared instance — several separate pipelines (and two
tuning searches) are about to be fit, and each should own its own encoder rather
than silently re-fitting a shared one underneath the others.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

def make_preprocessor():
    return ColumnTransformer(
        [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
        remainder="passthrough",
    )

## 3. Model

*Baseline first, then every model this dataset has seen so far, then this
lesson's two new ones — at their defaults, then tuned.*

Seven models in all, every one fit on the identical split above: the Week 2
majority-class baseline, Week 3's logistic regression, Lesson 3's pruned tree,
and this lesson's **Random Forest** and **XGBoost**, each fit twice — once with
the textbook defaults from Lessons 5-6, and once tuned in **Model Selection**
below.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

`C` below was chosen ahead of time by 5-fold cross-validation on the training
rows, scored by AUC over a grid of `C` values, and is hard-coded here the same
way the pruned tree below reuses Lesson 3's `ccp_alpha`.
`C` is the inverse of Week 3 Lesson 9's ridge penalty, so a smaller `C` means a
stronger penalty. Cross-validation landed close to scikit-learn's default of
1, so this scores essentially the same as Week 3's untuned fit: with about
5,600 training customers and 46 columns, there's little overfitting for a
penalty to fix.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logreg = Pipeline([
    ("prep", make_preprocessor()),
    # L2 (ridge) is LogisticRegression's default penalty; C=1.78 sets its strength
    ("clf", LogisticRegression(C=1.78, max_iter=5000, random_state=RANDOM_SEED)),
]).fit(X_train, y_train)

`ccp_alpha` below is Lesson 3's cross-validated answer, not a fresh search —
same split, same alpha, same tree that notebook already found. Re-deriving it
here would just repeat Lesson 3's `GridSearchCV`; the point of this notebook is
the two new models.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

pruned_tree = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", DecisionTreeClassifier(ccp_alpha=0.000986, random_state=RANDOM_SEED)),
]).fit(X_train, y_train)

**Random Forest**: deep, unpruned trees (Lesson 5's reversal of Lesson 2's
pruning instinct), bagged and decorrelated by restricting each split to a random
subset of features. `n_estimators=300` is comfortably past where Lesson 5's
OOB-error curves flattened — B is a compute-budget choice here, not something to
cross-validate. `oob_score=True` gets Lesson 5's free validation estimate as a
byproduct of fitting. These are Lesson 5's textbook settings; Model Selection
below checks whether they're actually right for this data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", RandomForestClassifier(n_estimators=300, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1)),
]).fit(X_train, y_train)

print(f"Random Forest OOB accuracy: {random_forest.named_steps['clf'].oob_score_:.2%}")

**XGBoost**: shallow trees fit sequentially to the ensemble's residual/gradient
(Lesson 6), each one's contribution shrunk by the learning rate. `max_depth=3`
and `learning_rate=0.1` are reasonable defaults, not a tuned answer — Model
Selection below chooses them deliberately instead of assuming them. Not part of the
shared Setup cell's packages, since it's the first lesson that needs it — a
short, lesson-specific install here instead.

In [ ]:
%pip install -q xgboost
from xgboost import XGBClassifier

xgboost_model = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=RANDOM_SEED)),
]).fit(X_train, y_train)

### Model Selection

*Lesson 7's discipline, applied: choose hyperparameters by held-out AUC on the
training rows only. The test set stays untouched until Evaluate.*

**Why AUC, when Lesson 3 tuned on accuracy?** Lesson 3's reason was specific to
a single tree: a 13-leaf tree produces only 13 distinct churn scores, too
coarse for a ranking metric to work with. An ensemble averages or sums hundreds
of trees, so its scores are fine-grained, and Week 2's warning applies in full.
Accuracy only checks which side of a 50% cutoff each customer lands on, a
threshold the business hasn't chosen (Week 2 set it from the cost of a missed
churner vs. a wasted call). AUC grades what the retention team actually uses: how
well the model *ranks* customers by risk, whatever cutoff they pick later. It's
also the metric Evaluate reports.

**Random Forest** has two knobs worth checking. `max_features` is Lesson 5's
*m*: how many features each split is allowed to consider. `min_samples_leaf`
is the smallest number of training customers a leaf may hold. At the default
of 1, every tree keeps splitting until its leaves hold single customers —
Lesson 5's "deep, unpruned" trees. Twelve combinations is small enough to try
every one.

There's no need for cross-validation here. Lesson 5's **out-of-bag** score
already *is* a held-out estimate: each tree skips about a third of the training
rows, and every row is scored only by the trees that never saw it. So each
combination needs one forest, not five — one fit per setting, scored by its own
OOB AUC. (The XGBoost search below takes a few minutes on Colab.)

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterGrid

# One forest per combination, each scored by its own out-of-bag AUC
oob_results = []
for params in ParameterGrid({
    "min_samples_leaf": [1, 5, 10, 20],
    "max_features": ["sqrt", 0.2, 0.5],
}):
    forest = Pipeline([
        ("prep", make_preprocessor()),
        ("clf", RandomForestClassifier(n_estimators=300, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1, **params)),
    ]).fit(X_train, y_train)
    # Each training row's churn probability from only the trees that never saw it
    oob_proba = forest.named_steps["clf"].oob_decision_function_[:, 1]
    oob_results.append({**params, "oob_auc": roc_auc_score(y_train, oob_proba), "model": forest})

oob_results = pd.DataFrame(oob_results)
best = oob_results.loc[oob_results["oob_auc"].idxmax()]
random_forest_tuned = best["model"]  # already fit on all of X_train, so no refit needed
print(f"Best settings: min_samples_leaf={best['min_samples_leaf']}, max_features={best['max_features']}")

# OOB AUC for every combination: rows = minimum leaf size, columns = features per split
oob_results.pivot_table(
    index="min_samples_leaf",
    columns=oob_results["max_features"].astype(str),
    values="oob_auc",
).round(4)

Read the table down the rows first. Moving from single-customer leaves
(`min_samples_leaf=1`, OOB AUC about 0.826) to a minimum of 5-20 customers per
leaf lifts OOB AUC by about 0.02 or more, in every column, and the gain has
mostly flattened out by 20. Across each row, `max_features` matters far less
(within about 0.006). So the knob doing the work here isn't Lesson 5's
random-feature trick. It's how far each tree is allowed to grow.

That refines Lesson 5 rather than overturning it. "Grow deep trees and let
averaging handle the variance" is right in spirit: the winning trees are still
far deeper than anything boosting uses. But churn is noisy: customers with
identical-looking profiles go both ways. A tree grown down to single-customer
leaves memorizes that noise, and because every tree in the forest trains on
mostly the same noisy rows, averaging doesn't fully cancel it out. Light
pruning does.

**XGBoost** has far more knobs — Lesson 7's map: step size and round count,
tree shape, randomization, and the three leaf penalties (γ, λ, α). That's too
many to try every combination, so this is Lesson 7's **randomized search plus
early stopping**: 100 random combinations drawn from the space below, each
scored by 5-fold CV AUC. Boosting has no out-of-bag rows to fall back on — every
round trains on the whole training set, or a subsample that changes each round
— so cross-validation it is.

The number of rounds isn't in the search space. Each candidate gets **early
stopping** instead: XGBoost's own `xgb.cv` keeps adding rounds while log-loss on
the held-out folds is still improving, and stops once 20 rounds pass without a
real improvement (a drop of more than 0.0001, so slow candidates can't keep
creeping forward on tiny gains). That's "stop at the bottom of Lesson 6's curve," found separately for
every candidate — a slow learning rate earns more rounds, a fast one fewer. It's
also Week 2's loss-vs-metric split: rounds stop on the loss the model trains
on, candidates are compared on the metric the business reads (AUC). Column sampling is `colsample_bynode`, a fresh random
fraction of features at every split: Lesson 5's random-feature trick, exactly as
Random Forest uses it.

One practical note: `xgb.cv` works on already-encoded data, so the one-hot
encoder is fit on the training rows once, outside a Pipeline. That's safe here:
it only learns which category names exist, and the test set is still untouched.
The final model goes back into a Pipeline like every other one.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import ParameterSampler

# xgb.cv needs encoded rows; the encoder only learns category names, so fitting it once on X_train is safe
dtrain = xgb.DMatrix(make_preprocessor().fit_transform(X_train), label=y_train)

xgb_results = []
for params in ParameterSampler({
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4, 5, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bynode": [0.6, 0.8, 1.0],
    "gamma": [0, 0.5, 1, 2, 5],
    "reg_lambda": [0.1, 1, 5, 10, 20],
    "reg_alpha": [0, 0.1, 1],
}, n_iter=100, random_state=RANDOM_SEED):
    # 5-fold CV that stops adding rounds once held-out log-loss hasn't improved by 0.0001 for 20 rounds
    history = xgb.cv(
        {**params, "objective": "binary:logistic", "seed": RANDOM_SEED},
        dtrain,
        num_boost_round=2000,
        nfold=5,
        stratified=True,
        metrics=["auc", "logloss"],
        seed=RANDOM_SEED,
        callbacks=[xgb.callback.EarlyStopping(rounds=20, metric_name="logloss", data_name="test", min_delta=1e-4)],
    )
    xgb_results.append({**params, "n_estimators": len(history), "cv_auc": history["test-auc-mean"].iloc[-1]})

xgb_best = max(xgb_results, key=lambda r: r["cv_auc"])
best_params = {name: value for name, value in xgb_best.items() if name != "cv_auc"}

# Refit the winner on all of X_train, with early stopping's round count as n_estimators
xgboost_tuned = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", XGBClassifier(**best_params, random_state=RANDOM_SEED)),
]).fit(X_train, y_train)

print(f"Best CV AUC: {xgb_best['cv_auc']:.4f}")
best_params

The winner is a textbook boosting configuration, and the opposite shape from
Random Forest's: shallow trees (`max_depth=3`) and a moderate learning rate
(`learning_rate=0.05`), run for the 128 rounds early stopping chose. It leans
hard on randomization: each round trains on a random 60% of the rows
(`subsample`), and each split considers a random 60% of the features
(`colsample_bynode`). Its one penalty is a full-strength L1 penalty
(`reg_alpha=1`); λ stays at XGBoost's default of 1, and γ isn't used at all.
Shallow, randomized, and lightly penalized: Lesson 7's levers, pulled in the
direction it predicted.

Don't over-read the exact winner, though. The top five candidates sit within
about 0.0003 of each other in CV AUC, with round counts from 124 to 612. Early
stopping matched each candidate's rounds to its learning rate, which is why fast
and slow settings ended up in a near-tie. Shallow trees (depth 2-3) and row
subsampling are what the leaders have in common.

## 4. Evaluate

*Every model, scored against the same test set — a metric only means something
next to what it is beating.*

In [ ]:
from sklearn.metrics import accuracy_score

models = {
    "Baseline (majority class)": baseline,
    "Logistic regression (Wk 3)": logreg,
    "Pruned tree (Lesson 3)": pruned_tree,
    "Random Forest, defaults (Lesson 5)": random_forest,
    "Random Forest, tuned": random_forest_tuned,
    "XGBoost, defaults (Lesson 6)": xgboost_model,
    "XGBoost, tuned": xgboost_tuned,
}

accuracy = pd.Series({name: accuracy_score(y_test, m.predict(X_test)) for name, m in models.items()})
accuracy = accuracy.sort_values()
accuracy

In [ ]:
colors = ["#9AA0A6"] * len(accuracy)
colors[-1] = "#1A73E8"
accuracy.plot.barh(color=colors, title="Test accuracy by model")
plt.xlabel("Accuracy")
plt.tight_layout()

Tuning helped both ensembles, but by different amounts. **Random Forest gained
just over two points** (77.6% → 79.8%), almost entirely from the leaf-size fix
in Model Selection, enough to go from trailing the single pruned tree to second
place. **XGBoost gained about a point** (79.2% → 80.1%): on 1,409 test
customers, that's 13 more correct predictions, enough to take first place four
customers ahead of tuned Random Forest, but small enough that a different split
could easily reorder the leaders. XGBoost's defaults (shallow trees, a modest
learning rate) were already closer to right for this data. Random Forest's
default of fully grown trees was the one setting genuinely holding a model
back.

The bigger picture hasn't changed, though. Every model past the baseline lands
within about two and a half points of each other, and the top four (both
XGBoosts, tuned Random Forest, and logistic regression) sit within about a
point. Bagging and
boosting are real fixes for a tree's variance and bias, but "real fix" doesn't
mean "wins big on every dataset." Telco churn is a small, fairly clean table
where a handful of mostly-monotonic features (tenure, contract type, charges)
already explain most of what's explainable. That's close to logistic
regression's home turf, and it leaves little room for ensembling's usual
advantage on messier, higher-interaction data to show up.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

scores = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items() if name != "Baseline (majority class)"}
auc = pd.Series({name: roc_auc_score(y_test, s) for name, s in scores.items()}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
for name in auc.index:  # highest AUC first, so the legend reads top to bottom
    fpr, tpr, _ = roc_curve(y_test, scores[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc[name]:.4f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC — every model, same test set")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()

AUC tells the same story as the accuracy bars: logistic regression, tuned
Random Forest, and both XGBoosts sit within 0.008 of each other at the top
(0.827 to 0.834, all 0.83 to two decimals), while the pruned tree and Random
Forest at its defaults trail slightly (0.81). Logistic regression comes
second, just behind tuned XGBoost. Ranking customers
by risk, not just the default 50% vote, doesn't separate the leaders either —
and it confirms that tuning is what brought Random Forest up to the pack.

## 5. Interpret

*What is each model actually keying off of? Business-readable, not just
statistically significant.*

Neither ensemble hands back a single traceable path the way Lesson 3's tree
did, or a signed coefficient the way Week 3's logistic regression did — but
both report **feature importance**: roughly, how much each feature contributed
to reducing error across every tree in the ensemble. Treat this as a first
read, not the final word — Lesson 10 covers specific ways split-based
importance can mislead, and Lesson 11 builds the more rigorous SHAP-based
explanation on top of this notebook's tuned XGBoost. The charts below show both
tuned models — plus Random Forest at its defaults, for a reason the next cell
explains.

In [ ]:
def top_importance(model, n=8):
    importance = pd.Series(
        model.named_steps["clf"].feature_importances_,
        index=model.named_steps["prep"].get_feature_names_out(),
    )
    top = importance.sort_values().tail(n)
    top.index = [name.replace("encode__", "").replace("remainder__", "") for name in top.index]
    return top

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, model, title in (
    (axes[0], random_forest, "Random Forest, defaults"),
    (axes[1], random_forest_tuned, "Random Forest, tuned"),
    (axes[2], xgboost_tuned, "XGBoost, tuned"),
):
    top_importance(model).plot.barh(ax=ax, color="#1A73E8", title=title)
plt.tight_layout()

Two things to notice. First, the two tuned models still tell different
stories. Random Forest spreads importance across tenure, contract type,
`TotalCharges`, and a handful of service features, with `tenure` narrowly
ahead of `Contract_Month-to-month` at the top. XGBoost concentrates:
`Contract_Month-to-month` alone accounts for about a quarter of its total
importance, nearly three times the next feature (fiber-optic internet), and
`tenure` barely makes its top eight. Same dataset, test accuracy within a third
of a point, different internal account of "what matters."

Second, compare the two Random Forest charts. Changing one hyperparameter, the
minimum leaf size, reshuffled the ranking: at the defaults, `TotalCharges`,
`tenure`, and `MonthlyCharges` lead and contract type carries only about 5% of
the importance; tuned, contract type more than doubles and moves to second. Same
algorithm, same data, different story. That's a preview of Lesson 10's point:
importance rankings are artifacts of one specific fitted model, not a single
ground truth to read off. Fully grown trees in particular keep finding new
split points in continuous columns like `TotalCharges`, which inflates their
importance.

## 6. Recommend

**Recommendation:** tuning didn't change the story; it confirmed it. The top
four models sit within about a point of accuracy and share the same AUC
(0.83 to two decimals), so
accuracy alone doesn't settle which one to deploy. This is a case where
business criteria should legitimately break the tie. Week 3's **logistic
regression** gives up about a point to the best ensemble (15 of 1,409
customers) and is far easier to explain and maintain, which makes it a reasonable incumbent for this
particular dataset, not a consolation prize. If the team wants the last bit of
accuracy, the **tuned XGBoost** is the candidate to carry forward: it tops
the chart on both accuracy and AUC, a few customers ahead of tuned Random
Forest, and it's the model Lesson 11
explains customer by customer. Keep Lesson 3's single tree in the toolkit for
stakeholder-facing explanations of *why* a specific customer was flagged.

**One operating rule to take with you:** whichever ensemble you train, check
the tree-size settings before trusting the defaults. Random Forest's default
of fully grown trees cost two points of accuracy on this noisy dataset, and a
quick sweep over `min_samples_leaf`, scored by out-of-bag AUC, found it.

**What's still open:** Lesson 9 places these models on the interpretability
spectrum this notebook's importance charts only hinted at, Lesson 10 names
specific ways split-based importance (just seen shifting between models,
and between two settings of the same model) can mislead, and Lesson 11 builds
SHAP explanations for individual predictions on this notebook's tuned XGBoost —
the rigorous version of the "what is it keying off of" question asked here.